# AifaQuant 快速开始

本 Notebook 演示完整的 A股 AI 量化流程：数据获取 → 因子构建 → 模型训练 → 回测。

In [ ]:
import sys

sys.path.append("..")  # 如果从 notebooks 目录运行

from aifa_quant.backtest import BacktestEngine, compute_metrics
from aifa_quant.config.settings import Settings
from aifa_quant.data.adapters import StockMCPAdapter
from aifa_quant.data.storage import DuckDBStore
from aifa_quant.features import FeatureBuilder
from aifa_quant.models import LGBRankerModel
from aifa_quant.models.registry import ModelRegistry
from aifa_quant.strategy import TopKDropoutStrategy

settings = Settings()

## 1. 检查 iFind MCP 连接

In [ ]:
adapter = StockMCPAdapter(settings)
tools = adapter.list_tools()
print(f"发现 {len(tools)} 个工具")
for t in tools[:5]:
    print(t["name"], "-", t.get("description", "")[:50])

## 2. 查看本地数据

In [ ]:
store = DuckDBStore(settings)
df = store.load_daily_quotes()
print(f"股票数: {df['symbol'].nunique()}, 总记录: {len(df)}")
df.head()

## 3. 特征工程

In [ ]:
builder = FeatureBuilder(settings)
features = builder.build_features(start_date="20230101", end_date="20231231")
feature_cols = builder.feature_columns(features)
print(f"样本数: {len(features)}, 特征数: {len(feature_cols)}")
features[["symbol", "trade_date", "close", "label_return"] + feature_cols[:5]].head()

## 4. 训练模型

In [ ]:
X = features[feature_cols]
y = features["label_binary"]

model = LGBRankerModel()
model.fit(X, y, feature_cols)

print("Top 10 重要特征:")
print(model.feature_importance.head(10))

registry = ModelRegistry(settings)
model.save(str(registry.path("lgb_stock_selector")))

## 5. 回测

In [ ]:
test_features = builder.build_features(start_date="20240101", end_date="20241231")
test_features["pred_score"] = model.predict(test_features[feature_cols])

symbols = test_features["symbol"].unique().tolist()
quotes = store.load_daily_quotes(symbols, start_date="20240101", end_date="20241231")

strategy = TopKDropoutStrategy(top_k=3, rebalance_freq=5)
engine = BacktestEngine(initial_cash=1_000_000, rebalance_freq=5)
equity = engine.run(quotes, test_features, model, strategy)

metrics = compute_metrics(equity)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

## 6. 绘制权益曲线

In [ ]:
import matplotlib.pyplot as plt

equity["normalized"] = equity["total_value"] / equity["total_value"].iloc[0]
plt.figure(figsize=(12, 6))
plt.plot(equity["trade_date"], equity["normalized"])
plt.title("Strategy Equity Curve")
plt.xlabel("Date")
plt.ylabel("Normalized Value")
plt.grid(True)